Управляющий ноутбук будет:
- последовательно обрабатывать структуры, по одной за раз;
- проверять результаты каждого этапа;
- пропускать уже успешно обработанные структуры;
- повторять обработку неудачных структур;
- вести общий журнал ошибок и времени выполнения;
- собирать сводные таблицы по всем белкам и найденным карманам.
Важный принцип: сохраняем все обнаруженные карманы, а не только лучший. Отбор наиболее перспективных карманов будет отдельным этапом анализа.

In [1]:
# ============================================================
# CELL 1. ENVIRONMENT AND IMPORTS
# ============================================================
# Назначение:
# 1. Подготовка среды пакетной обработки.
# 2. Импорт библиотек.
# 3. Проверка необходимых модулей.
# 4. Настройка путей проекта.
# ============================================================

import os
import sys
import re
import json
import time
import shutil
import traceback
import warnings

from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

try:
    import nbformat
    from nbclient import NotebookClient
    from tqdm.auto import tqdm

except ImportError as exc:
    raise ImportError(
        "Не установлены необходимые библиотеки.\n"
        "Выполните один раз в отдельной ячейке:\n"
        "%pip install nbformat nbclient tqdm"
    ) from exc


# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(
    r"C:\apex\APEX\МИФИ\СНО\medic"
)

DATASET_ROOT = (
    PROJECT_ROOT / "Protein_Pocket_Dataset"
)

STRUCTURE_ANALYSIS_DIR = (
    DATASET_ROOT / "07_structure_analysis"
)

STRUCTURES_DIR = (
    STRUCTURE_ANALYSIS_DIR / "structures"
)

RESULTS_ROOT = (
    STRUCTURE_ANALYSIS_DIR / "results"
)

SELECTION_CSV = (
    DATASET_ROOT
    / "05_final_dataset"
    / "protein_final_100.csv"
)

# Локальный путь к исходному ноутбуку.
ANALYZER_NOTEBOOK_PATH = (
    PROJECT_ROOT / "Protein Structure Analyzer.ipynb"
)

# Каталог пакетного управления.
BATCH_ROOT = (
    STRUCTURE_ANALYSIS_DIR / "batch_runs"
)

BATCH_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 70)
print("BATCH PROTEIN PROCESSOR")
print("=" * 70)

print("Python:", sys.version)
print("Project root:", PROJECT_ROOT)
print("Analyzer notebook:", ANALYZER_NOTEBOOK_PATH)
print("Selection CSV:", SELECTION_CSV)
print("Results root:", RESULTS_ROOT)

print("=" * 70)

# Проверки путей.

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"PROJECT_ROOT не существует:\n{PROJECT_ROOT}"
    )

if not SELECTION_CSV.is_file():
    raise FileNotFoundError(
        f"Не найден CSV отбора:\n{SELECTION_CSV}"
    )

if not ANALYZER_NOTEBOOK_PATH.is_file():
    raise FileNotFoundError(
        f"Не найден исходный ноутбук:\n"
        f"{ANALYZER_NOTEBOOK_PATH}\n\n"
        "Исправьте ANALYZER_NOTEBOOK_PATH "
        "на фактический локальный путь."
    )

print("Environment validation: OK")

BATCH PROTEIN PROCESSOR
Python: 3.10.6 (tags/v3.10.6:9c7b4bd, Aug  1 2022, 21:53:49) [MSC v.1932 64 bit (AMD64)]
Project root: C:\apex\APEX\МИФИ\СНО\medic
Analyzer notebook: C:\apex\APEX\МИФИ\СНО\medic\Protein Structure Analyzer.ipynb
Selection CSV: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\05_final_dataset\protein_final_100.csv
Results root: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\results
Environment validation: OK


In [2]:

# ============================================================
# CELL 2. BATCH CONFIGURATION
# ============================================================
# Назначение:
# 1. Выбор режима пакетной обработки.
# 2. Настройка количества PDB.
# 3. Настройка возобновления.
# 4. Настройка времени выполнения.
# 5. Автоматическое определение Jupyter Kernel.
# 6. Настройка целевых компонентов по PDB.
# ============================================================


# ------------------------------------------------------------
# 1. PROCESSING MODE
# ------------------------------------------------------------

# TEST  — первые N структур.
# BATCH — весь список или выбранное подмножество.

BATCH_MODE = "BATCH"

ALLOWED_BATCH_MODES = {
    "TEST",
    "BATCH"
}

if BATCH_MODE not in ALLOWED_BATCH_MODES:
    raise ValueError(
        f"Недопустимый BATCH_MODE: {BATCH_MODE}"
    )


# ------------------------------------------------------------
# 2. PDB SELECTION
# ------------------------------------------------------------

TEST_SIZE = 5

# None — использовать весь список из CSV.
# Или указать конкретные PDB:
# CUSTOM_PDB_IDS = ["1OCC", "1ABC"]

CUSTOM_PDB_IDS = None

# Начальный индекс в исходном списке.
BATCH_START_INDEX = 0

# Максимальное количество структур.
# None — без ограничения.

BATCH_LIMIT = None


# ------------------------------------------------------------
# 3. RESUME SETTINGS
# ------------------------------------------------------------

# Пропускать ранее успешно обработанные PDB,
# если итоговые файлы существуют.

SKIP_COMPLETED = True

# Повторять структуры, ранее завершившиеся ошибкой.

RETRY_FAILED = True


# ------------------------------------------------------------
# 4. EXECUTION SETTINGS
# ------------------------------------------------------------

# Максимальное время выполнения одной ячейки
# исходного ноутбука, секунды.

CELL_TIMEOUT = 3600

# Имя ядра определяется автоматически
# из metadata исходного ноутбука.

KERNEL_NAME = None

if not ANALYZER_NOTEBOOK_PATH.exists():
    raise FileNotFoundError(
        "Исходный ноутбук не найден:\n"
        f"{ANALYZER_NOTEBOOK_PATH}"
    )

# Чтение metadata исходного ноутбука.

import nbformat

analyzer_metadata = nbformat.read(
    ANALYZER_NOTEBOOK_PATH,
    as_version=4
).metadata

notebook_kernel = (
    analyzer_metadata
    .get("kernelspec", {})
    .get("name")
)

if KERNEL_NAME is None:
    KERNEL_NAME = notebook_kernel

if not isinstance(KERNEL_NAME, str) or not KERNEL_NAME.strip():
    raise ValueError(
        "Не удалось определить Jupyter Kernel.\n"
        "Проверь metadata исходного ноутбука "
        "или укажи KERNEL_NAME вручную."
    )

KERNEL_NAME = KERNEL_NAME.strip()

# Сохранять исполненные копии исходного ноутбука.
# Для 100 структур может занимать много места.

SAVE_EXECUTED_NOTEBOOK = False


# ------------------------------------------------------------
# 5. BATCH RUN IDENTIFIER
# ------------------------------------------------------------

# Имя должно оставаться неизменным,
# если необходимо продолжить прежний запуск.

BATCH_RUN_NAME = "protein_batch_v1"

BATCH_RUN_DIR = (
    BATCH_ROOT / BATCH_RUN_NAME
)

BATCH_RUN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BATCH_LOG_DIR = (
    BATCH_RUN_DIR / "logs"
)

BATCH_NOTEBOOK_DIR = (
    BATCH_RUN_DIR / "executed_notebooks"
)

BATCH_LOG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

if SAVE_EXECUTED_NOTEBOOK:
    BATCH_NOTEBOOK_DIR.mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# 6. COMPONENT SELECTION REGISTRY
# ------------------------------------------------------------

# Выбор целевого небелкового компонента задаётся
# индивидуально для каждого PDB.
#
# Не следует автоматически назначать HEM всем белкам.
#
# Пример:
#
# COMPONENT_SELECTION_BY_PDB = {
#     "1OCC": {
#         "component_name": "HEA",
#         "chain": "A",
#         "residue_number": None,
#         "component_index": None
#     }
# }
#
# Если PDB отсутствует в словаре:
# автоматический выбор компонента отключается.
#
# Это не мешает остальному анализу структуры.

COMPONENT_SELECTION_BY_PDB = {}


# ------------------------------------------------------------
# 7. CONFIGURATION VALIDATION
# ------------------------------------------------------------

if not isinstance(TEST_SIZE, int) or TEST_SIZE < 1:
    raise ValueError("TEST_SIZE должен быть целым числом >= 1.")

if not isinstance(BATCH_START_INDEX, int) or BATCH_START_INDEX < 0:
    raise ValueError("BATCH_START_INDEX должен быть целым числом >= 0.")

if BATCH_LIMIT is not None:
    if not isinstance(BATCH_LIMIT, int) or BATCH_LIMIT < 1:
        raise ValueError("BATCH_LIMIT должен быть None или целым числом >= 1.")

if CELL_TIMEOUT is not None:
    if not isinstance(CELL_TIMEOUT, int) or CELL_TIMEOUT < 1:
        raise ValueError("CELL_TIMEOUT должен быть положительным числом.")

if not isinstance(BATCH_RUN_NAME, str) or not BATCH_RUN_NAME.strip():
    raise ValueError("BATCH_RUN_NAME не может быть пустым.")

if not isinstance(COMPONENT_SELECTION_BY_PDB, dict):
    raise TypeError("COMPONENT_SELECTION_BY_PDB должен быть словарём.")


# ------------------------------------------------------------
# 8. CONFIGURATION SUMMARY
# ------------------------------------------------------------

print("=" * 70)
print("BATCH CONFIGURATION")
print("=" * 70)

print("Mode:", BATCH_MODE)
print("Test size:", TEST_SIZE)
print("Custom PDB list:", CUSTOM_PDB_IDS)
print("Start index:", BATCH_START_INDEX)
print("Batch limit:", BATCH_LIMIT)

print("-" * 70)

print("Skip completed:", SKIP_COMPLETED)
print("Retry failed:", RETRY_FAILED)
print("Cell timeout:", CELL_TIMEOUT)
print("Kernel:", KERNEL_NAME)

print("-" * 70)

print("Analyzer notebook:", ANALYZER_NOTEBOOK_PATH)
print("Kernel metadata:", notebook_kernel)

print("-" * 70)

print("Component registry entries:",
      len(COMPONENT_SELECTION_BY_PDB))

print("Batch run:", BATCH_RUN_NAME)
print("Batch directory:", BATCH_RUN_DIR)

print("-" * 70)

print("Environment validation: OK")
print("=" * 70)


BATCH CONFIGURATION
Mode: BATCH
Test size: 5
Custom PDB list: None
Start index: 0
Batch limit: None
----------------------------------------------------------------------
Skip completed: True
Retry failed: True
Cell timeout: 3600
Kernel: python3
----------------------------------------------------------------------
Analyzer notebook: C:\apex\APEX\МИФИ\СНО\medic\Protein Structure Analyzer.ipynb
Kernel metadata: python3
----------------------------------------------------------------------
Component registry entries: 0
Batch run: protein_batch_v1
Batch directory: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\batch_runs\protein_batch_v1
----------------------------------------------------------------------
Environment validation: OK


In [3]:

# ============================================================
# CELL 3. PDB REGISTRY AND PROCESSING STATE
# ============================================================
# Назначение:
# 1. Загрузка списка PDB.
# 2. Проверка идентификаторов.
# 3. Формирование списка обработки.
# 4. Загрузка предыдущих статусов.
# 5. Подготовка реестра результатов.
# ============================================================


# ------------------------------------------------------------
# 1. LOAD PDB REGISTRY
# ------------------------------------------------------------

if not SELECTION_CSV.is_file():
    raise FileNotFoundError(
        f"Selection CSV не найден:\n{SELECTION_CSV}"
    )

selection_df = pd.read_csv(
    SELECTION_CSV,
    dtype={"pdb_id": "string"}
)

if "pdb_id" not in selection_df.columns:
    raise ValueError(
        "В CSV отсутствует столбец pdb_id."
    )

selection_df["pdb_id"] = (
    selection_df["pdb_id"]
    .astype("string")
    .str.strip()
    .str.upper()
)

selection_df = selection_df.dropna(
    subset=["pdb_id"]
).copy()

selection_df = selection_df[
    selection_df["pdb_id"] != ""
].copy()

if selection_df.empty:
    raise ValueError(
        "После очистки CSV не осталось PDB ID."
    )


# ------------------------------------------------------------
# 2. SORT REGISTRY
# ------------------------------------------------------------

if "selection_order" in selection_df.columns:

    selection_df["selection_order"] = pd.to_numeric(
        selection_df["selection_order"],
        errors="coerce"
    )

    selection_df = selection_df.sort_values(
        "selection_order",
        na_position="last"
    )

selection_df = selection_df.reset_index(
    drop=True
)


# ------------------------------------------------------------
# 3. VALIDATE PDB IDS
# ------------------------------------------------------------

invalid_ids = [
    pdb_id
    for pdb_id in selection_df["pdb_id"].astype(str)
    if not re.fullmatch(r"[A-Z0-9]{4}", pdb_id)
]

if invalid_ids:
    raise ValueError(
        f"Некорректные PDB ID: {invalid_ids[:10]}"
    )

if selection_df["pdb_id"].duplicated().any():

    duplicates = (
        selection_df.loc[
            selection_df["pdb_id"].duplicated(keep=False),
            "pdb_id"
        ]
        .tolist()
    )

    raise ValueError(
        f"Повторяющиеся PDB ID: {duplicates[:10]}"
    )


# ------------------------------------------------------------
# 4. BUILD PROCESSING LIST
# ------------------------------------------------------------

AVAILABLE_PDB_IDS = (
    selection_df["pdb_id"]
    .astype(str)
    .tolist()
)

if CUSTOM_PDB_IDS is not None:

    PDB_IDS = [
        str(x).strip().upper()
        for x in CUSTOM_PDB_IDS
    ]

    if len(PDB_IDS) != len(set(PDB_IDS)):
        raise ValueError(
            "CUSTOM_PDB_IDS содержит повторяющиеся PDB ID."
        )

    invalid_custom_ids = [
        x for x in PDB_IDS
        if not re.fullmatch(r"[A-Z0-9]{4}", x)
    ]

    if invalid_custom_ids:
        raise ValueError(
            f"Некорректные CUSTOM PDB ID: {invalid_custom_ids}"
        )

    unknown_ids = [
        x for x in PDB_IDS
        if x not in AVAILABLE_PDB_IDS
    ]

    if unknown_ids:
        raise ValueError(
            "PDB отсутствуют в selection CSV: "
            f"{unknown_ids}"
        )

else:

    PDB_IDS = AVAILABLE_PDB_IDS.copy()


# ------------------------------------------------------------
# 5. APPLY PROCESSING LIMITS
# ------------------------------------------------------------

if not isinstance(BATCH_START_INDEX, int):
    raise TypeError(
        "BATCH_START_INDEX должен быть целым числом."
    )

if BATCH_START_INDEX < 0:
    raise ValueError(
        "BATCH_START_INDEX не может быть отрицательным."
    )

PDB_IDS = PDB_IDS[BATCH_START_INDEX:]


if BATCH_MODE == "TEST":
    PDB_IDS = PDB_IDS[:TEST_SIZE]


if BATCH_LIMIT is not None:
    PDB_IDS = PDB_IDS[:BATCH_LIMIT]


if not PDB_IDS:
    raise ValueError(
        "Список PDB для обработки пуст."
    )


# ------------------------------------------------------------
# 6. PROCESSING STATE FILES
# ------------------------------------------------------------

STATE_CSV = (
    BATCH_RUN_DIR / "batch_status.csv"
)

SUMMARY_CSV = (
    BATCH_RUN_DIR / "batch_summary.csv"
)

MANIFEST_JSON = (
    BATCH_RUN_DIR / "batch_manifest.json"
)


# ------------------------------------------------------------
# 7. LOAD PREVIOUS STATE
# ------------------------------------------------------------

STATE_COLUMNS = [
    "pdb_id",
    "status",
    "started_at",
    "finished_at",
    "duration_seconds",
    "output_dir",
    "error",
    "log_file"
]

if STATE_CSV.is_file():

    try:
        previous_state = pd.read_csv(
            STATE_CSV,
            dtype={"pdb_id": "string"}
        )

    except Exception as exc:
        raise RuntimeError(
            f"Не удалось прочитать файл состояния:\n"
            f"{STATE_CSV}\n"
            f"Причина: {exc}"
        ) from exc

    if "pdb_id" not in previous_state.columns:
        raise ValueError(
            "В batch_status.csv отсутствует столбец pdb_id."
        )

    previous_state["pdb_id"] = (
        previous_state["pdb_id"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

else:

    previous_state = pd.DataFrame(
        columns=STATE_COLUMNS
    )


# ------------------------------------------------------------
# 8. NORMALIZE PREVIOUS STATE
# ------------------------------------------------------------

previous_state = previous_state.dropna(
    subset=["pdb_id"]
).copy()

previous_state = previous_state[
    previous_state["pdb_id"] != ""
].copy()

# Последняя запись по каждому PDB.

if not previous_state.empty:

    previous_state = (
        previous_state
        .drop_duplicates(
            subset=["pdb_id"],
            keep="last"
        )
        .reset_index(drop=True)
    )


PREVIOUS_STATUS = {
    str(row["pdb_id"]): row.to_dict()
    for _, row in previous_state.iterrows()
}


# ------------------------------------------------------------
# 9. DISPLAY REGISTRY
# ------------------------------------------------------------

print("=" * 70)
print("PDB REGISTRY")
print("=" * 70)

print("Total available:", len(AVAILABLE_PDB_IDS))
print("Selected for processing:", len(PDB_IDS))

print("-" * 70)

for i, pdb_id in enumerate(PDB_IDS, start=1):
    print(f"{i:3d}. {pdb_id}")

print("-" * 70)

print("Previous records:", len(PREVIOUS_STATUS))

print("=" * 70)


PDB REGISTRY
Total available: 100
Selected for processing: 100
----------------------------------------------------------------------
  1. 7WTU
  2. 7W37
  3. 7Q0G
  4. 7VYS
  5. 7X8R
  6. 7QB5
  7. 7Z0O
  8. 7XCR
  9. 12TB
 10. 3PUW
 11. 7WTT
 12. 3O2R
 13. 3NV0
 14. 7Z16
 15. 7WSO
 16. 7PR4
 17. 7W38
 18. 3OFG
 19. 7XNX
 20. 3P1A
 21. 7W1Z
 22. 7OYK
 23. 7WIG
 24. 7PS6
 25. 7W2U
 26. 3NZJ
 27. 7XCT
 28. 7YMS
 29. 3NY7
 30. 3P0Y
 31. 7Y3G
 32. 7YE4
 33. 7P4R
 34. 7W5Z
 35. 7PCA
 36. 7WU2
 37. 3OS8
 38. 7YE6
 39. 3P3O
 40. 7Z13
 41. 7PEF
 42. 7XD0
 43. 7PU4
 44. 3OR7
 45. 7W2Y
 46. 3O0A
 47. 7XMT
 48. 3PC8
 49. 21ZV
 50. 3O1C
 51. 7Y7M
 52. 3PSN
 53. 7W4C
 54. 3NWS
 55. 7Z38
 56. 3POZ
 57. 7VZ4
 58. 3OZZ
 59. 7Y75
 60. 3P8V
 61. 7XCP
 62. 3O1X
 63. 7XW6
 64. 3NZL
 65. 7W4F
 66. 3OSX
 67. 21ZW
 68. 7Q8Z
 69. 7Q2T
 70. 7Y76
 71. 3P8Y
 72. 7X3W
 73. 3OCM
 74. 7WZ4
 75. 7OX3
 76. 7W9K
 77. 7OYI
 78. 7XSZ
 79. 3O2P
 80. 3P6E
 81. 7X6O
 82. 7P8R
 83. 7XOV
 84. 7YI0
 85. 3NUL
 86. 3OM1
 87. 7

In [4]:

# ============================================================
# CELL 4. EXECUTION FUNCTIONS
# ============================================================
# Назначение:
# 1. Подготовка исходного ноутбука.
# 2. Передача PDB_ID в CELL 2.
# 3. Индивидуальная настройка CELL 17.
# 4. Исключение повторной установки библиотек.
# 5. Определение Jupyter Kernel.
# 6. Запуск NotebookClient.
# 7. Проверка результатов и их актуальности.
# 8. Сохранение журнала выполнения.
# 9. Защита от ошибок сохранения вспомогательных файлов.
# 10. Поддержка проверки старых результатов без
#     обязательного повторного обновления файлов.
# ============================================================


# ------------------------------------------------------------
# 0. IMPORTS
# ------------------------------------------------------------

import re
import json
import time
import traceback

from datetime import datetime

import numpy as np
import pandas as pd
import nbformat

from nbclient import NotebookClient


# ------------------------------------------------------------
# 1. NOTEBOOK OUTPUT EXTRACTION
# ------------------------------------------------------------

def extract_notebook_output(notebook):

    output_parts = []

    for cell_index, cell in enumerate(notebook.cells):

        if cell.cell_type != "code":
            continue

        for output in cell.get("outputs", []):

            output_type = output.get("output_type")

            if output_type == "stream":

                output_parts.append(
                    str(output.get("text", ""))
                )

            elif output_type == "error":

                output_parts.append(
                    "\n".join(
                        output.get("traceback", [])
                    )
                )

            elif output_type in {
                "display_data",
                "execute_result"
            }:

                data = output.get("data", {})

                text_value = data.get(
                    "text/plain",
                    ""
                )

                if text_value:

                    output_parts.append(
                        str(text_value)
                    )

    return "\n".join(output_parts)


# ------------------------------------------------------------
# 2. SKIP INSTALLATION COMMANDS
# ------------------------------------------------------------

def skip_installation_cells(notebook):

    skipped = 0

    installation_patterns = (
        "%pip install",
        "!pip install",
        "%conda install",
        "!conda install"
    )

    for cell in notebook.cells:

        if cell.cell_type != "code":
            continue

        original_lines = cell.source.splitlines()

        filtered_lines = []

        cell_changed = False

        for line in original_lines:

            stripped = line.strip().lower()

            is_installation = any(
                stripped.startswith(pattern)
                for pattern in installation_patterns
            )

            if is_installation:

                filtered_lines.append(
                    "# Installation command skipped by Batch Processor."
                )

                cell_changed = True

            else:

                filtered_lines.append(line)

        if cell_changed:

            cell.source = "\n".join(filtered_lines)

            skipped += 1

    return skipped


# ------------------------------------------------------------
# 3. PATCH COMPONENT SELECTION
# ------------------------------------------------------------

def patch_component_selection(
    notebook,
    pdb_id
):

    config = COMPONENT_SELECTION_BY_PDB.get(
        pdb_id,
        {}
    )

    target_cells = []

    for cell in notebook.cells:

        if cell.cell_type != "code":
            continue

        source = cell.source

        if (
            "PREFERRED_COMPONENT_NAME =" in source
            and "SELECT_COMPONENT_INDEX =" in source
        ):

            target_cells.append(cell)

    if len(target_cells) != 1:

        raise RuntimeError(
            "Не удалось однозначно определить CELL 17.\n"
            f"Найдено подходящих ячеек: {len(target_cells)}.\n"
            "Проверь настройки выбора компонента "
            "в исходном ноутбуке."
        )

    target_cell = target_cells[0]

    replacements = {

        "PREFERRED_COMPONENT_NAME":
            repr(config.get("component_name")),

        "PREFERRED_COMPONENT_CHAIN":
            repr(config.get("chain")),

        "PREFERRED_COMPONENT_RESIDUE_NUMBER":
            repr(config.get("residue_number")),

        "SELECT_COMPONENT_INDEX":
            repr(config.get("component_index"))
    }

    source = target_cell.source

    for variable_name, value in replacements.items():

        pattern = (
            rf"(?m)^([ \t]*){re.escape(variable_name)}"
            rf"\s*=\s*.*$"
        )

        source, count = re.subn(
            pattern,
            lambda match: (
                match.group(1)
                + variable_name
                + " = "
                + value
            ),
            source
        )

        if count != 1:

            raise RuntimeError(
                f"Не удалось заменить {variable_name} "
                f"в настройках CELL 17.\n"
                f"Количество совпадений: {count}.\n"
                "Проверь актуальный код CELL 17."
            )

    target_cell.source = source

    return config


# ------------------------------------------------------------
# 4. RESOLVE NOTEBOOK KERNEL
# ------------------------------------------------------------

def resolve_notebook_kernel(notebook):

    kernel_name = KERNEL_NAME

    if not kernel_name:

        kernel_name = (
            notebook.metadata
            .get("kernelspec", {})
            .get("name")
        )

    if not isinstance(kernel_name, str):

        raise RuntimeError(
            "Не удалось определить Jupyter Kernel.\n"
            "Проверь KERNEL_NAME и metadata исходного ноутбука."
        )

    kernel_name = kernel_name.strip()

    if not kernel_name:

        raise RuntimeError(
            "Имя Jupyter Kernel пустое."
        )

    return kernel_name


# ------------------------------------------------------------
# 5. PREPARE NOTEBOOK FOR ONE PDB
# ------------------------------------------------------------

def prepare_notebook_for_pdb(pdb_id):

    if not ANALYZER_NOTEBOOK_PATH.is_file():

        raise FileNotFoundError(
            f"Исходный ноутбук не найден:\n"
            f"{ANALYZER_NOTEBOOK_PATH}"
        )

    notebook = nbformat.read(
        ANALYZER_NOTEBOOK_PATH,
        as_version=4
    )

    # Проверяем ядро до запуска.

    notebook_kernel = resolve_notebook_kernel(
        notebook
    )

    # Удаляем только команды установки.

    skipped_install_cells = (
        skip_installation_cells(notebook)
    )

    # Настраиваем целевой компонент.

    component_config = patch_component_selection(
        notebook,
        pdb_id
    )

    # Передаём параметры до выполнения CELL 2.

    parameter_source = f"""
# ============================================================
# BATCH PROCESSOR PARAMETERS
# ============================================================

PDB_ID_INPUT = {pdb_id!r}

PDB_ID_OVERRIDE = None

BATCH_COMPONENT_SELECTION = {component_config!r}

print("Batch PDB ID:", PDB_ID_INPUT)
print("Component configuration:", BATCH_COMPONENT_SELECTION)
print("Notebook kernel:", {notebook_kernel!r})
"""

    parameter_cell = nbformat.v4.new_code_cell(
        source=parameter_source
    )

    parameter_cell.metadata["tags"] = [
        "batch_parameters"
    ]

    notebook.cells.insert(
        0,
        parameter_cell
    )

    return (
        notebook,
        skipped_install_cells,
        notebook_kernel
    )


# ------------------------------------------------------------
# 6. OUTPUT FILE SIGNATURE
# ------------------------------------------------------------

def get_file_signature(path):

    if path is None:

        return None

    if not path.is_file():

        return None

    stat = path.stat()

    return (
        stat.st_size,
        stat.st_mtime_ns
    )


# ------------------------------------------------------------
# 7. REQUIRED OUTPUT VALIDATION
# ------------------------------------------------------------

def validate_pdb_outputs(
    pdb_id,
    previous_signatures=None,
    require_fresh=True
):

    pdb_output_dir = RESULTS_ROOT / pdb_id

    if not pdb_output_dir.exists():

        return {

            "valid": False,

            "output_dir": str(pdb_output_dir),

            "graph_file": None,

            "features_json": None,

            "graph_valid": False,

            "features_valid": False,

            "graph_fresh": False,

            "features_fresh": False,

            "graph_candidates_count": 0,

            "features_candidates_count": 0,

            "message": "PDB results directory not found"
        }

    # --------------------------------------------------------
    # Поиск обязательных файлов
    # --------------------------------------------------------

    graph_candidates = sorted(
        pdb_output_dir.rglob(
            f"{pdb_id}_ml_graph_data.npz"
        )
    )

    feature_candidates = sorted(
        pdb_output_dir.rglob(
            f"{pdb_id}_protein_features.json"
        )
    )

    # При неоднозначности файл не выбирается.

    graph_file = (
        graph_candidates[0]
        if len(graph_candidates) == 1
        else None
    )

    features_json = (
        feature_candidates[0]
        if len(feature_candidates) == 1
        else None
    )

    graph_valid = False

    features_valid = False

    graph_fresh = False

    features_fresh = False

    graph_error = ""

    features_error = ""

    # --------------------------------------------------------
    # NPZ VALIDATION
    # --------------------------------------------------------

    if graph_file is not None:

        try:

            graph_signature = get_file_signature(
                graph_file
            )

            if graph_signature is None:

                raise FileNotFoundError(
                    f"NPZ file not found: {graph_file}"
                )

            if graph_signature[0] <= 0:

                raise ValueError(
                    "NPZ file is empty."
                )

            # Проверка актуальности.
            #
            # require_fresh=False:
            # допускается использование ранее
            # сохранённого корректного файла.
            #
            # require_fresh=True:
            # требуется изменение сигнатуры
            # относительно состояния до запуска.

            if previous_signatures is None:

                graph_fresh = not require_fresh

            else:

                old_signature = previous_signatures.get(
                    str(graph_file)
                )

                graph_fresh = (
                    graph_signature != old_signature
                )

            # Проверяем возможность чтения архива
            # и каждого массива без использования pickle.

            with np.load(
                graph_file,
                allow_pickle=False
            ) as graph_data:

                graph_keys = list(
                    graph_data.files
                )

                if not graph_keys:

                    raise ValueError(
                        "NPZ archive contains no arrays."
                    )

                for key in graph_keys:

                    array = graph_data[key]

                    if not isinstance(
                        array,
                        np.ndarray
                    ):

                        raise TypeError(
                            f"Invalid NPZ array: {key}"
                        )

            graph_valid = True

        except Exception as exc:

            graph_error = (
                f"{type(exc).__name__}: {exc}"
            )

    else:

        graph_error = (
            "Expected exactly one NPZ file, "
            f"found {len(graph_candidates)}"
        )

    # --------------------------------------------------------
    # JSON VALIDATION
    # --------------------------------------------------------

    if features_json is not None:

        try:

            features_signature = get_file_signature(
                features_json
            )

            if features_signature is None:

                raise FileNotFoundError(
                    f"JSON file not found: {features_json}"
                )

            if features_signature[0] <= 0:

                raise ValueError(
                    "JSON file is empty."
                )

            # Проверка актуальности.

            if previous_signatures is None:

                features_fresh = not require_fresh

            else:

                old_signature = previous_signatures.get(
                    str(features_json)
                )

                features_fresh = (
                    features_signature != old_signature
                )

            # Проверяем корректность JSON.

            with open(
                features_json,
                "r",
                encoding="utf-8"
            ) as file:

                features_data = json.load(file)

            if not isinstance(
                features_data,
                dict
            ):

                raise ValueError(
                    "Features JSON must contain a dictionary."
                )

            features_valid = True

        except Exception as exc:

            features_error = (
                f"{type(exc).__name__}: {exc}"
            )

    else:

        features_error = (
            "Expected exactly one JSON file, "
            f"found {len(feature_candidates)}"
        )

    # --------------------------------------------------------
    # FINAL VALIDATION
    # --------------------------------------------------------

    unique_files = (
        len(graph_candidates) == 1
        and len(feature_candidates) == 1
    )

    # Оба файла должны быть корректными.
    # При require_fresh=True оба должны обновиться.
    # При require_fresh=False старые корректные
    # файлы допускаются без обновления.

    freshness_ok = (

        not require_fresh

        or (

            graph_fresh

            and features_fresh
        )
    )

    valid = (

        unique_files

        and graph_valid

        and features_valid

        and freshness_ok
    )

    messages = []

    if len(graph_candidates) != 1:

        messages.append(
            f"Expected one NPZ, found {len(graph_candidates)}"
        )

    if len(feature_candidates) != 1:

        messages.append(
            f"Expected one JSON, found {len(feature_candidates)}"
        )

    if not graph_valid:

        messages.append(
            f"Invalid NPZ: {graph_error}"
        )

    if not features_valid:

        messages.append(
            f"Invalid JSON: {features_error}"
        )

    if require_fresh and graph_valid and not graph_fresh:

        messages.append(
            "NPZ was not updated during current run"
        )

    if require_fresh and features_valid and not features_fresh:

        messages.append(
            "JSON was not updated during current run"
        )

    return {

        "valid": bool(valid),

        "output_dir": str(pdb_output_dir),

        "graph_file": (
            str(graph_file)
            if graph_file is not None
            else None
        ),

        "features_json": (
            str(features_json)
            if features_json is not None
            else None
        ),

        "graph_valid": bool(graph_valid),

        "features_valid": bool(features_valid),

        "graph_fresh": bool(graph_fresh),

        "features_fresh": bool(features_fresh),

        "graph_candidates_count": len(graph_candidates),

        "features_candidates_count": len(feature_candidates),

        "message": (

            "Required outputs found, readable and updated"

            if valid and require_fresh

            else (

                "Existing required outputs found and readable"

                if valid

                else "; ".join(messages)
            )
        )
    }


# ------------------------------------------------------------
# 8. ATOMIC STATE SAVE
# ------------------------------------------------------------

def save_batch_state(records):

    df = pd.DataFrame(records)

    temporary_path = STATE_CSV.with_suffix(
        ".tmp.csv"
    )

    df.to_csv(
        temporary_path,
        index=False,
        encoding="utf-8-sig"
    )

    # Атомарная замена основного файла состояния.

    temporary_path.replace(
        STATE_CSV
    )


# ------------------------------------------------------------
# 9. RUN ONE PDB
# ------------------------------------------------------------

def run_one_pdb(pdb_id):

    started_at = datetime.now()

    start_time = time.perf_counter()

    log_path = (
        BATCH_LOG_DIR / f"{pdb_id}.log.txt"
    )

    executed_notebook_path = (
        BATCH_NOTEBOOK_DIR / f"{pdb_id}.ipynb"
    )

    status = "FAILED"

    error_message = ""

    notebook_output = ""

    output_validation = {}

    skipped_install_cells = 0

    notebook = None

    notebook_kernel = None

    previous_signatures = {}

    notebook_saved = False

    log_saved = False

    try:

        # ----------------------------------------------------
        # Сохранение состояния файлов ДО запуска.
        # ----------------------------------------------------

        pdb_output_dir = RESULTS_ROOT / pdb_id

        if pdb_output_dir.exists():

            for path in pdb_output_dir.rglob("*"):

                if path.is_file():

                    previous_signatures[str(path)] = (
                        get_file_signature(path)
                    )

        print(
            f"\nPreparing notebook for {pdb_id}..."
        )

        # ----------------------------------------------------
        # Подготовка ноутбука.
        # ----------------------------------------------------

        (
            notebook,
            skipped_install_cells,
            notebook_kernel
        ) = prepare_notebook_for_pdb(
            pdb_id
        )

        print(
            f"[{pdb_id}] Kernel: {notebook_kernel}"
        )

        print(
            f"[{pdb_id}] Skipped installation cells: "
            f"{skipped_install_cells}"
        )

        # ----------------------------------------------------
        # Создание независимого Jupyter Kernel.
        # ----------------------------------------------------

        client = NotebookClient(

            notebook,

            timeout=CELL_TIMEOUT,

            kernel_name=notebook_kernel,

            allow_errors=False,

            resources={

                "metadata": {

                    "path": str(PROJECT_ROOT)
                }
            }
        )

        print(
            f"Starting independent kernel: {pdb_id}"
        )

        # ----------------------------------------------------
        # Выполнение исходного ноутбука.
        # ----------------------------------------------------

        client.execute()

        # ----------------------------------------------------
        # Извлечение вывода.
        # ----------------------------------------------------

        notebook_output = (
            extract_notebook_output(notebook)
        )

        # ----------------------------------------------------
        # Проверка обязательных результатов.
        # Оба файла должны быть обновлены после запуска.
        # ----------------------------------------------------

        output_validation = validate_pdb_outputs(

            pdb_id,

            previous_signatures=previous_signatures,

            require_fresh=True
        )

        if output_validation["valid"]:

            status = "SUCCESS"

            error_message = ""

        else:

            status = "FAILED_OUTPUT_VALIDATION"

            error_message = (
                output_validation["message"]
            )

    except Exception:

        status = "FAILED"

        error_message = traceback.format_exc()

        if notebook is not None:

            try:

                notebook_output = (
                    extract_notebook_output(notebook)
                )

            except Exception as output_exc:

                notebook_output = (
                    "Не удалось извлечь вывод ноутбука: "
                    f"{type(output_exc).__name__}: "
                    f"{output_exc}"
                )

    finally:

        finished_at = datetime.now()

        duration = (
            time.perf_counter() - start_time
        )

        # ----------------------------------------------------
        # Сохранение выполненного ноутбука.
        # ----------------------------------------------------

        if (

            SAVE_EXECUTED_NOTEBOOK

            and notebook is not None
        ):

            try:

                BATCH_NOTEBOOK_DIR.mkdir(
                    parents=True,
                    exist_ok=True
                )

                nbformat.write(
                    notebook,
                    executed_notebook_path
                )

                notebook_saved = True

                print(
                    f"[{pdb_id}] Executed notebook saved."
                )

            except Exception as exc:

                save_error = (

                    "Не удалось сохранить выполненный ноутбук: "

                    f"{type(exc).__name__}: {exc}"
                )

                error_message = (

                    f"{error_message}\n{save_error}"

                    if error_message

                    else save_error
                )

                print(
                    f"[{pdb_id}] WARNING: {save_error}"
                )

        # ----------------------------------------------------
        # Формирование журнала выполнения.
        # ----------------------------------------------------

        log_text = f"""
============================================================
BATCH PROCESSING LOG
============================================================

PDB ID: {pdb_id}

Started: {started_at.isoformat()}
Finished: {finished_at.isoformat()}

Duration: {duration:.2f} seconds

Status: {status}

Kernel: {notebook_kernel}

Skipped installation cells:
{skipped_install_cells}

Executed notebook saved:
{notebook_saved}

Output validation:
{json.dumps(output_validation, ensure_ascii=False, indent=4)}

============================================================
NOTEBOOK OUTPUT
============================================================

{notebook_output}

============================================================
ERROR
============================================================

{error_message}
"""

        # ----------------------------------------------------
        # Устойчивое сохранение журнала.
        # ----------------------------------------------------

        try:

            BATCH_LOG_DIR.mkdir(
                parents=True,
                exist_ok=True
            )

            log_path.write_text(
                log_text,
                encoding="utf-8"
            )

            log_saved = True

        except Exception as exc:

            log_error = (

                "Не удалось сохранить лог: "

                f"{type(exc).__name__}: {exc}"
            )

            error_message = (

                f"{error_message}\n{log_error}"

                if error_message

                else log_error
            )

            print(
                f"[{pdb_id}] WARNING: {log_error}"
            )

    # --------------------------------------------------------
    # Результат обработки одной структуры.
    # --------------------------------------------------------

    return {

        "pdb_id": pdb_id,

        "status": status,

        "started_at": started_at.isoformat(),

        "finished_at": finished_at.isoformat(),

        "duration_seconds": round(duration, 2),

        "output_dir": str(
            RESULTS_ROOT / pdb_id
        ),

        "graph_file": output_validation.get(
            "graph_file"
        ),

        "features_json": output_validation.get(
            "features_json"
        ),

        "error": error_message,

        "log_file": (
            str(log_path)
            if log_saved
            else None
        ),

        "notebook_file": (
            str(executed_notebook_path)
            if notebook_saved
            else None
        ),

        "log_saved": log_saved,

        "notebook_saved": notebook_saved
    }


print("Execution functions initialized.")


Execution functions initialized.


In [5]:

# ============================================================
# CELL 5. BATCH EXECUTION ENGINE
# ============================================================
# Назначение:
# 1. Последовательная обработка PDB.
# 2. Пропуск успешно обработанных структур.
# 3. Повтор неудачных запусков по настройке.
# 4. Регистрация времени и ошибок.
# 5. Сохранение состояния после каждого PDB.
# 6. Исключение дублирующихся записей в истории.
# 7. Восстановление состояния после прерывания.
# 8. Итоговая статистика выполнения.
# ============================================================


# ------------------------------------------------------------
# 1. RESTORE PREVIOUS RECORDS
# ------------------------------------------------------------

batch_records = []

for _, row in previous_state.iterrows():

    batch_records.append(
        row.to_dict()
    )


# ------------------------------------------------------------
# 2. CURRENT STATUS MAP
# ------------------------------------------------------------

def get_latest_status_map(records):

    result = {}

    for record in records:

        pdb_id = str(
            record.get("pdb_id", "")
        ).strip().upper()

        if not pdb_id:
            continue

        record = dict(record)

        record["pdb_id"] = pdb_id

        # Последняя запись для PDB заменяет предыдущую.
        result[pdb_id] = record

    return result


status_map = get_latest_status_map(
    batch_records
)

# Убираем дубли из истории.
batch_records = list(
    status_map.values()
)


# ------------------------------------------------------------
# 2.1. NORMALIZE AND DEDUPLICATE PDB IDS
# ------------------------------------------------------------

PDB_IDS_UNIQUE = list(
    dict.fromkeys(
        str(pdb_id).strip().upper()
        for pdb_id in PDB_IDS
        if str(pdb_id).strip()
    )
)

print(
    "Unique PDB count:",
    len(PDB_IDS_UNIQUE)
)

if len(PDB_IDS_UNIQUE) != len(PDB_IDS):

    print(
        "WARNING: Duplicate or empty PDB IDs were removed."
    )


# ------------------------------------------------------------
# 3. BATCH EXECUTION
# ------------------------------------------------------------

batch_start_time = time.perf_counter()

batch_started_at = datetime.now()

execution_results = []

batch_interrupted = False

current_pdb_id = None

print("=" * 75)
print("BATCH EXECUTION STARTED")
print("=" * 75)

print("PDB count:", len(PDB_IDS_UNIQUE))
print("Mode:", BATCH_MODE)
print("Kernel:", KERNEL_NAME)
print("Run directory:", BATCH_RUN_DIR)
print("Skip completed:", SKIP_COMPLETED)
print("Retry failed:", RETRY_FAILED)

print("=" * 75)


try:

    for pdb_id in tqdm(
        PDB_IDS_UNIQUE,
        desc="Processing PDB structures",
        unit="PDB"
    ):

        pdb_id = str(
            pdb_id
        ).strip().upper()

        current_pdb_id = pdb_id

        previous_record = status_map.get(
            pdb_id
        )

        previous_status = (

            str(
                previous_record.get("status", "")
            ).strip().upper()

            if previous_record is not None

            else ""
        )

        # ----------------------------------------------------
        # CHECK PREVIOUS SUCCESS
        # ----------------------------------------------------

        if (

            SKIP_COMPLETED

            and previous_record is not None

            and previous_status == "SUCCESS"
        ):

            # Для уже обработанных структур не требуется
            # обновление файлов в текущем запуске.
            # Проверяется только наличие и корректность файлов.

            validation = validate_pdb_outputs(
                pdb_id,
                require_fresh=False
            )

            if validation["valid"]:

                print(
                    f"\n[{pdb_id}] SKIPPED: already completed."
                )

                execution_results.append({

                    **previous_record,

                    "status": "SKIPPED_COMPLETED"
                })

                # Не изменяем status_map:
                # в постоянной истории остаётся SUCCESS.

                continue

            else:

                print(
                    f"\n[{pdb_id}] Previous SUCCESS has invalid outputs."
                )

                print(
                    "The structure will be processed again."
                )

                print(
                    "Validation:",
                    validation["message"]
                )

        # ----------------------------------------------------
        # CHECK PREVIOUS FAILURE
        # ----------------------------------------------------

        if (

            previous_record is not None

            and previous_status.startswith("FAILED")

            and not RETRY_FAILED
        ):

            print(
                f"\n[{pdb_id}] SKIPPED: previous failure."
            )

            execution_results.append({

                **previous_record,

                "status": "SKIPPED_PREVIOUS_FAILURE"
            })

            # Сохраняем исходный FAILED в истории.
            continue

        # ----------------------------------------------------
        # RUN CURRENT PDB
        # ----------------------------------------------------

        print("\n" + "=" * 75)

        print(
            f"PROCESSING: {pdb_id}"
        )

        print("=" * 75)

        # ----------------------------------------------------
        # Защита основного цикла от ошибки одной структуры.
        # ----------------------------------------------------

        try:

            result = run_one_pdb(
                pdb_id
            )

        except Exception:

            # Дополнительная защита на случай,
            # если run_one_pdb не смог вернуть результат.

            error_message = traceback.format_exc()

            result = {

                "pdb_id": pdb_id,

                "status": "FAILED",

                "started_at": datetime.now().isoformat(),

                "finished_at": datetime.now().isoformat(),

                "duration_seconds": 0.0,

                "output_dir": str(
                    RESULTS_ROOT / pdb_id
                ),

                "graph_file": None,

                "features_json": None,

                "error": error_message,

                "log_file": None,

                "notebook_file": None,

                "log_saved": False,

                "notebook_saved": False
            }

            print(
                f"[{pdb_id}] ERROR: run_one_pdb failed."
            )

            print(error_message)

        # ----------------------------------------------------
        # REGISTER RESULT
        # ----------------------------------------------------

        execution_results.append(
            result
        )

        # Обновляем актуальную запись.
        status_map[pdb_id] = result

        # Формируем историю без дублирующихся PDB.
        batch_records = list(
            status_map.values()
        )

        # ----------------------------------------------------
        # SAVE STATE AFTER EACH PDB
        # ----------------------------------------------------

        try:

            save_batch_state(
                batch_records
            )

            print(
                f"[{pdb_id}] Batch state saved."
            )

        except Exception:

            state_error = traceback.format_exc()

            print(
                f"[{pdb_id}] CRITICAL: state save failed."
            )

            print(state_error)

            # Не продолжаем большой batch,
            # если невозможно сохранить состояние.
            raise RuntimeError(
                f"Не удалось сохранить состояние после {pdb_id}."
            )

        # ----------------------------------------------------
        # PRINT CURRENT RESULT
        # ----------------------------------------------------

        print(
            f"\n[{pdb_id}] STATUS: {result['status']}"
        )

        print(
            f"Duration: {result['duration_seconds']} sec"
        )

        if result["status"] != "SUCCESS":

            print(
                f"Log: {result.get('log_file')}"
            )

            if result.get("error"):

                print(
                    "Error:",
                    result["error"]
                )


except KeyboardInterrupt:

    batch_interrupted = True

    print("\n" + "=" * 75)

    print("BATCH INTERRUPTED BY USER")

    print("=" * 75)

    # Уже завершённые структуры остаются в state.
    # Текущая незавершённая структура при следующем
    # запуске будет обработана повторно.


except Exception:

    batch_interrupted = True

    print("\n" + "=" * 75)

    print("BATCH STOPPED DUE TO CRITICAL ERROR")

    print("=" * 75)

    print(
        traceback.format_exc()
    )


finally:

    # --------------------------------------------------------
    # 4. FINAL EXECUTION STATE
    # --------------------------------------------------------

    batch_finished_at = datetime.now()

    batch_duration = (
        time.perf_counter() - batch_start_time
    )

    # Финальная история состояния.
    batch_records = list(
        status_map.values()
    )

    batch_state_df = pd.DataFrame(
        batch_records
    )

    # Финальное атомарное сохранение.
    try:

        save_batch_state(
            batch_records
        )

        print(
            "\nFinal batch state saved."
        )

    except Exception:

        print(
            "\nCRITICAL: final state save failed."
        )

        print(
            traceback.format_exc()
        )

    # --------------------------------------------------------
    # 5. EXECUTION SUMMARY
    # --------------------------------------------------------

    execution_summary = pd.DataFrame(
        execution_results
    )

    if not execution_summary.empty:

        summary_counts = (
            execution_summary["status"]
            .value_counts()
        )

    else:

        summary_counts = pd.Series(
            dtype=int
        )

    print("\n" + "=" * 75)

    if batch_interrupted:

        print(
            "BATCH EXECUTION STOPPED"
        )

    else:

        print(
            "BATCH EXECUTION FINISHED"
        )

    print("=" * 75)

    print(
        "Started:",
        batch_started_at
    )

    print(
        "Finished:",
        batch_finished_at
    )

    print(
        f"Total duration: {batch_duration / 60:.2f} minutes"
    )

    print(
        "Total unique PDB IDs:",
        len(PDB_IDS_UNIQUE)
    )

    print(
        "Entries in current run:",
        len(execution_results)
    )

    print("\nCURRENT RUN STATUS:")

    if not summary_counts.empty:

        print(
            summary_counts.to_string()
        )

    else:

        print(
            "No structures processed."
        )

    print(
        "\nTOTAL STRUCTURES IN STATE:",
        len(batch_state_df)
    )

    print(
        "State file:",
        STATE_CSV
    )

    print("=" * 75)


Unique PDB count: 100
BATCH EXECUTION STARTED
PDB count: 100
Mode: BATCH
Kernel: python3
Run directory: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\batch_runs\protein_batch_v1
Skip completed: True
Retry failed: True


Processing PDB structures:   0%|          | 0/100 [00:00<?, ?PDB/s]


[7WTU] SKIPPED: already completed.

[7W37] SKIPPED: already completed.

[7Q0G] SKIPPED: already completed.

[7VYS] SKIPPED: already completed.

[7X8R] SKIPPED: already completed.

[7QB5] SKIPPED: already completed.

[7Z0O] SKIPPED: already completed.

[7XCR] SKIPPED: already completed.

[12TB] SKIPPED: already completed.

[3PUW] SKIPPED: already completed.

[7WTT] SKIPPED: already completed.

[3O2R] SKIPPED: already completed.

[3NV0] SKIPPED: already completed.

[7Z16] SKIPPED: already completed.

[7WSO] SKIPPED: already completed.

[7PR4] SKIPPED: already completed.

[7W38] SKIPPED: already completed.

[3OFG] SKIPPED: already completed.

[7XNX] SKIPPED: already completed.

[3P1A] SKIPPED: already completed.

[7W1Z] SKIPPED: already completed.

[7OYK] SKIPPED: already completed.

[7WIG] SKIPPED: already completed.

[7PS6] SKIPPED: already completed.

[7W2U] SKIPPED: already completed.

[3NZJ] SKIPPED: already completed.

[7XCT] SKIPPED: already completed.

[7YMS] SKIPPED: already com

In [6]:
# ============================================================
# CELL 6. BATCH QUALITY CONTROL
# ============================================================
# Назначение:
# 1. Проверка итоговых файлов каждого PDB.
# 2. Проверка наличия NPZ графа.
# 3. Проверка JSON с признаками белка.
# 4. Формирование итоговой таблицы качества.
# 5. Сохранение результатов QC.
# ============================================================


qc_records = []

for pdb_id in PDB_IDS:

    output_check = validate_pdb_outputs(
        pdb_id
    )

    current_status = status_map.get(
        pdb_id,
        {}
    )

    qc_records.append({

        "pdb_id": pdb_id,

        "processing_status": current_status.get(
            "status",
            "NOT_PROCESSED"
        ),

        "graph_file_exists": (
            output_check["graph_file"] is not None
        ),

        "features_json_exists": (
            output_check["features_json"] is not None
        ),

        "outputs_valid": output_check["valid"],

        "output_dir": output_check["output_dir"],

        "graph_file": output_check["graph_file"],

        "features_json": output_check["features_json"],

        "qc_message": output_check["message"]

    })


df_batch_qc = pd.DataFrame(
    qc_records
)


# ------------------------------------------------------------
# SAVE QC
# ------------------------------------------------------------

QC_CSV = (
    BATCH_RUN_DIR / "batch_quality_control.csv"
)

df_batch_qc.to_csv(
    QC_CSV,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# STATUS COUNTS
# ------------------------------------------------------------

status_counts = (
    df_batch_qc["processing_status"]
    .value_counts(dropna=False)
    .to_dict()
)

valid_count = int(
    df_batch_qc["outputs_valid"].sum()
)

invalid_count = (
    len(df_batch_qc) - valid_count
)


print("=" * 75)
print("BATCH QUALITY CONTROL")
print("=" * 75)

print("Total structures:", len(df_batch_qc))
print("Valid output sets:", valid_count)
print("Invalid output sets:", invalid_count)

print("-" * 75)

print("PROCESSING STATUS")

for status, count in status_counts.items():

    print(
        f"{str(status):35s}: {count}"
    )

print("-" * 75)

display(
    df_batch_qc
)

print("-" * 75)

print("QC file:", QC_CSV)

print("=" * 75)

BATCH QUALITY CONTROL
Total structures: 100
Valid output sets: 0
Invalid output sets: 100
---------------------------------------------------------------------------
PROCESSING STATUS
SUCCESS                            : 100
---------------------------------------------------------------------------


,pdb_id,processing_status,graph_file_exists,features_json_exists,outputs_valid,output_dir,graph_file,features_json,qc_message
0,7WTU,SUCCESS,True,True,False,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,NPZ was not updated during current run; JSON w...
1,7W37,SUCCESS,True,True,False,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,NPZ was not updated during current run; JSON w...
2,7Q0G,SUCCESS,True,True,False,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,NPZ was not updated during current run; JSON w...
3,7VYS,SUCCESS,True,True,False,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,NPZ was not updated during current run; JSON w...
4,7X8R,SUCCESS,True,True,False,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,NPZ was not updated during current run; JSON w...
...,...,...,...,...,...,...,...,...,...
95,7WU3,SUCCESS,True,True,False,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,NPZ was not updated during current run; JSON w...
96,3PAM,SUCCESS,True,True,False,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,NPZ was not updated during current run; JSON w...
97,7XD1,SUCCESS,True,True,False,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,NPZ was not updated during current run; JSON w...
98,7P0J,SUCCESS,True,True,False,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,NPZ was not updated during current run; JSON w...


---------------------------------------------------------------------------
QC file: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\batch_runs\protein_batch_v1\batch_quality_control.csv


In [7]:

# ============================================================
# CELL 7. FINAL BATCH REPORT
# ============================================================
# Назначение:
# 1. Формирование итогового реестра пакетной обработки.
# 2. Подсчёт успешно обработанных структур.
# 3. Учёт ранее завершённых и пропущенных структур.
# 4. Подсчёт ошибок.
# 5. Оценка полноты выходных файлов.
# 6. Сохранение CSV и JSON-манифеста.
# ============================================================


# ============================================================
# 1. PREPARE FINAL QC DATA
# ============================================================

df_batch_final = df_batch_qc.copy()

total_structures = len(df_batch_final)

if total_structures == 0:
    raise ValueError(
        "Итоговая таблица QC пуста. "
        "Невозможно сформировать отчёт."
    )


# ============================================================
# 2. NORMALIZE STATUS VALUES
# ============================================================

df_batch_final["processing_status"] = (
    df_batch_final["processing_status"]
    .fillna("UNKNOWN")
    .astype(str)
    .str.strip()
    .str.upper()
)

df_batch_final["outputs_valid"] = (
    df_batch_final["outputs_valid"]
    .fillna(False)
    .astype(bool)
)


# ============================================================
# 3. CALCULATE PROCESSING STATISTICS
# ============================================================

# SUCCESS — успешно обработаны в текущем или
# предыдущем запуске.

# SKIPPED_COMPLETED — ранее обработаны и пропущены
# при текущем запуске.

# SKIPPED_PREVIOUS_FAILURE — не запускались повторно,
# поскольку RETRY_FAILED = False.

completed_statuses = {
    "SUCCESS",
    "SKIPPED_COMPLETED"
}

successful_structures = int(
    df_batch_final["processing_status"]
    .isin(completed_statuses)
    .sum()
)

skipped_completed = int(
    (
        df_batch_final["processing_status"]
        == "SKIPPED_COMPLETED"
    ).sum()
)

skipped_previous_failure = int(
    (
        df_batch_final["processing_status"]
        == "SKIPPED_PREVIOUS_FAILURE"
    ).sum()
)

failed_structures = int(
    df_batch_final["processing_status"]
    .str.startswith("FAILED")
    .sum()
)

not_processed = int(
    (
        df_batch_final["processing_status"]
        == "NOT_PROCESSED"
    ).sum()
)

unknown_status = int(
    (
        df_batch_final["processing_status"]
        == "UNKNOWN"
    ).sum()
)


# ============================================================
# 4. OUTPUT VALIDATION STATISTICS
# ============================================================

valid_outputs = int(
    df_batch_final["outputs_valid"].sum()
)

invalid_outputs = (
    total_structures - valid_outputs
)

success_rate = (
    successful_structures / total_structures * 100
)

output_validity_rate = (
    valid_outputs / total_structures * 100
)


# ============================================================
# 5. DURATION STATISTICS
# ============================================================

if "duration_seconds" in batch_state_df.columns:

    duration_values = pd.to_numeric(
        batch_state_df["duration_seconds"],
        errors="coerce"
    )

    duration_values = duration_values.dropna()

else:

    duration_values = pd.Series(
        dtype=float
    )


if not duration_values.empty:

    mean_duration = float(
        duration_values.mean()
    )

    median_duration = float(
        duration_values.median()
    )

    max_duration = float(
        duration_values.max()
    )

    min_duration = float(
        duration_values.min()
    )

else:

    mean_duration = None
    median_duration = None
    max_duration = None
    min_duration = None


# ============================================================
# 6. BUILD STATUS DISTRIBUTION
# ============================================================

status_distribution = (
    df_batch_final["processing_status"]
    .value_counts(dropna=False)
    .to_dict()
)

status_distribution = {
    str(key): int(value)
    for key, value in status_distribution.items()
}


# ============================================================
# 7. BUILD FINAL STRUCTURE REGISTRY
# ============================================================

# Объединяем сведения QC с актуальным состоянием
# пакетной обработки.

df_batch_registry = df_batch_final.copy()

df_batch_registry["batch_run_name"] = (
    BATCH_RUN_NAME
)

df_batch_registry["batch_mode"] = (
    BATCH_MODE
)

df_batch_registry["report_generated_at"] = (
    datetime.now().isoformat()
)


# ============================================================
# 8. SAVE FINAL CSV
# ============================================================

df_batch_registry.to_csv(
    SUMMARY_CSV,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 9. BUILD JSON MANIFEST
# ============================================================

manifest = {

    "batch_run_name": BATCH_RUN_NAME,

    "batch_mode": BATCH_MODE,

    "started_at": batch_started_at.isoformat(),

    "finished_at": batch_finished_at.isoformat(),

    "report_generated_at": datetime.now().isoformat(),

    "total_duration_seconds": round(
        float(batch_duration),
        2
    ),

    "processing_statistics": {

        "total_structures": total_structures,

        "successful_structures": successful_structures,

        "skipped_completed": skipped_completed,

        "skipped_previous_failure": skipped_previous_failure,

        "failed_structures": failed_structures,

        "not_processed": not_processed,

        "unknown_status": unknown_status,

        "success_rate_percent": round(
            success_rate,
            2
        )

    },

    "output_validation": {

        "valid_output_sets": valid_outputs,

        "invalid_output_sets": invalid_outputs,

        "output_validity_rate_percent": round(
            output_validity_rate,
            2
        )

    },

    "duration_statistics": {

        "mean_seconds": mean_duration,

        "median_seconds": median_duration,

        "min_seconds": min_duration,

        "max_seconds": max_duration

    },

    "status_distribution": status_distribution,

    "source_notebook": str(
        ANALYZER_NOTEBOOK_PATH
    ),

    "selection_csv": str(
        SELECTION_CSV
    ),

    "results_root": str(
        RESULTS_ROOT
    ),

    "batch_run_directory": str(
        BATCH_RUN_DIR
    ),

    "processed_pdb_ids": [
        str(x)
        for x in PDB_IDS
    ],

    "component_selection_registry": (
        COMPONENT_SELECTION_BY_PDB
    ),

    "output_files": {

        "state_csv": str(STATE_CSV),

        "quality_control_csv": str(QC_CSV),

        "summary_csv": str(SUMMARY_CSV),

        "manifest_json": str(MANIFEST_JSON)

    },

    "limitations": [

        "This batch run executes Protein Structure Analyzer only.",

        "Pocket Detection and Protein-Pocket Integration "
        "are not yet connected to this controller.",

        "Component selection requires a per-PDB registry.",

        "Output-file validation confirms file presence and "
        "non-zero file size, not scientific correctness.",

        "A skipped structure is considered completed only "
        "when its required output files pass validation."

    ]

}


# ============================================================
# 10. JSON SAFE CONVERSION
# ============================================================

def batch_json_safe(value):

    if isinstance(value, dict):

        return {
            str(key): batch_json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):

        return [
            batch_json_safe(item)
            for item in value
        ]

    if isinstance(value, Path):

        return str(value)

    if isinstance(value, (np.integer,)):

        return int(value)

    if isinstance(value, (np.floating,)):

        return (
            float(value)
            if np.isfinite(value)
            else None
        )

    if isinstance(value, (np.bool_,)):

        return bool(value)

    if isinstance(value, float):

        return (
            value
            if np.isfinite(value)
            else None
        )

    if value is pd.NA or value is pd.NaT:

        return None

    if isinstance(value, (datetime,)):

        return value.isoformat()

    return value


manifest = batch_json_safe(
    manifest
)


# ============================================================
# 11. SAVE JSON MANIFEST
# ============================================================

temporary_manifest = (
    MANIFEST_JSON.with_suffix(".tmp.json")
)

with open(
    temporary_manifest,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        manifest,
        file,
        ensure_ascii=False,
        indent=4,
        allow_nan=False
    )

temporary_manifest.replace(
    MANIFEST_JSON
)


# ============================================================
# 12. FINAL REPORT
# ============================================================

print("\n" + "=" * 75)
print("FINAL BATCH PROCESSING REPORT")
print("=" * 75)

print(f"Batch run:                 {BATCH_RUN_NAME}")
print(f"Processing mode:           {BATCH_MODE}")

print("-" * 75)

print(f"Total structures:          {total_structures}")
print(f"Successful structures:     {successful_structures}")
print(f"Skipped completed:         {skipped_completed}")
print(f"Skipped previous failure:  {skipped_previous_failure}")
print(f"Failed structures:         {failed_structures}")
print(f"Not processed:             {not_processed}")
print(f"Unknown status:            {unknown_status}")

print("-" * 75)

print(f"Valid output sets:          {valid_outputs}")
print(f"Invalid output sets:        {invalid_outputs}")

print(f"Success rate:               {success_rate:.2f}%")
print(f"Output validity rate:       {output_validity_rate:.2f}%")

print("-" * 75)

if mean_duration is not None:

    print(
        f"Mean PDB duration:          {mean_duration:.2f} sec"
    )

    print(
        f"Median PDB duration:        {median_duration:.2f} sec"
    )

    print(
        f"Minimum PDB duration:       {min_duration:.2f} sec"
    )

    print(
        f"Maximum PDB duration:       {max_duration:.2f} sec"
    )

else:

    print("Duration statistics:       unavailable")


print("-" * 75)

print("OUTPUT FILES")

print("State CSV:       ", STATE_CSV)
print("QC CSV:          ", QC_CSV)
print("Summary CSV:     ", SUMMARY_CSV)
print("Manifest JSON:   ", MANIFEST_JSON)

print("=" * 75)


# ============================================================
# 13. FINAL STATUS
# ============================================================

if failed_structures > 0:

    print(
        "WARNING: обнаружены структуры с ошибками."
    )

elif invalid_outputs > 0:

    print(
        "WARNING: часть структур не прошла проверку выходных файлов."
    )

elif not_processed > 0:

    print(
        "WARNING: имеются необработанные структуры."
    )

else:

    print(
        "BATCH PROCESSING COMPLETED."
    )


print("=" * 75)


# ============================================================
# 14. DISPLAY FINAL TABLE
# ============================================================

display(
    df_batch_registry
)



FINAL BATCH PROCESSING REPORT
Batch run:                 protein_batch_v1
Processing mode:           BATCH
---------------------------------------------------------------------------
Total structures:          100
Successful structures:     100
Skipped completed:         0
Skipped previous failure:  0
Failed structures:         0
Not processed:             0
Unknown status:            0
---------------------------------------------------------------------------
Valid output sets:          0
Invalid output sets:        100
Success rate:               100.00%
Output validity rate:       0.00%
---------------------------------------------------------------------------
Mean PDB duration:          43.07 sec
Median PDB duration:        17.96 sec
Minimum PDB duration:       10.42 sec
Maximum PDB duration:       1422.66 sec
---------------------------------------------------------------------------
OUTPUT FILES
State CSV:        C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_

,pdb_id,processing_status,graph_file_exists,features_json_exists,outputs_valid,output_dir,graph_file,features_json,qc_message,batch_run_name,batch_mode,report_generated_at
0,7WTU,SUCCESS,True,True,False,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,NPZ was not updated during current run; JSON w...,protein_batch_v1,BATCH,2026-11-07T14:46:28.159102
1,7W37,SUCCESS,True,True,False,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,NPZ was not updated during current run; JSON w...,protein_batch_v1,BATCH,2026-11-07T14:46:28.159102
2,7Q0G,SUCCESS,True,True,False,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,NPZ was not updated during current run; JSON w...,protein_batch_v1,BATCH,2026-11-07T14:46:28.159102
3,7VYS,SUCCESS,True,True,False,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,NPZ was not updated during current run; JSON w...,protein_batch_v1,BATCH,2026-11-07T14:46:28.159102
4,7X8R,SUCCESS,True,True,False,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,NPZ was not updated during current run; JSON w...,protein_batch_v1,BATCH,2026-11-07T14:46:28.159102
...,...,...,...,...,...,...,...,...,...,...,...,...
95,7WU3,SUCCESS,True,True,False,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,NPZ was not updated during current run; JSON w...,protein_batch_v1,BATCH,2026-11-07T14:46:28.159102
96,3PAM,SUCCESS,True,True,False,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,NPZ was not updated during current run; JSON w...,protein_batch_v1,BATCH,2026-11-07T14:46:28.159102
97,7XD1,SUCCESS,True,True,False,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,NPZ was not updated during current run; JSON w...,protein_batch_v1,BATCH,2026-11-07T14:46:28.159102
98,7P0J,SUCCESS,True,True,False,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,NPZ was not updated during current run; JSON w...,protein_batch_v1,BATCH,2026-11-07T14:46:28.159102
